# Telco Customer Churn Prediction - Model Training & Evaluation
## Phases 5, 6, and 7: Model Development, Evaluation, and Tuning

This notebook implements a comprehensive machine learning workflow including:

1. **Phase 5**: Train 6 classification models
2. **Phase 6**: Evaluate all models with comprehensive metrics
3. **Phase 7**: Perform hyperparameter tuning on top models

### Models to Train
1. Logistic Regression (Baseline)
2. Decision Tree (Baseline)
3. Random Forest (Baseline + Tuned)
4. K-Nearest Neighbors (Baseline)
5. Support Vector Machine (Baseline)
6. XGBoost (Baseline + Tuned)

### Evaluation Metrics
- Accuracy, Precision, Recall, F1 Score, ROC-AUC
- Confusion Matrix for each model
- Comparative visualizations across all models

In [ ]:
# Import libraries
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve)
import joblib
import os
import warnings

warnings.filterwarnings('ignore')

# Import custom utilities
from utils import (load_preprocessed_data, train_models, evaluate_models, 
                   hyperparameter_tuning, save_models, create_results_leaderboard)

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Create output directory
os.makedirs('visuals', exist_ok=True)
os.makedirs('models', exist_ok=True)

print("✓ All libraries imported and environment configured!")

## Step 1: Load Preprocessed Data

Load the training and test datasets that were preprocessed in notebook 01_EDA_Preprocessing.ipynb.
These datasets have already been:
- Feature engineered (One-Hot Encoded)
- Scaled (StandardScaler applied)
- Split into train (80%) and test (20%) sets
- Stratified to maintain class distribution

In [ ]:
# Load preprocessed data
X_train, X_test, y_train, y_test = load_preprocessed_data('data/processed')

print("\n" + "="*60)
print("DATA SUMMARY")
print("="*60)
print(f"\nTraining Data:")
print(f"  Features: {X_train.shape[1]}")
print(f"  Samples: {X_train.shape[0]:,}")
print(f"  Target distribution:\n{y_train.value_counts()}")

print(f"\nTest Data:")
print(f"  Features: {X_test.shape[1]}")
print(f"  Samples: {X_test.shape[0]:,}")
print(f"  Target distribution:\n{y_test.value_counts()}")

# Check data types
print(f"\nData Types - All numeric: {X_train.dtypes.unique()}")

## Phase 5: Train Classification Models

We'll train 6 different classification models with default hyperparameters as baselines.
These models represent different learning paradigms:

1. **Logistic Regression**: Linear model, interpretable, baseline classifier
2. **Decision Tree**: Tree-based, non-linear, prone to overfitting
3. **Random Forest**: Ensemble of trees, robust, good generalization
4. **K-Nearest Neighbors**: Instance-based, simple, can be slow on large data
5. **Support Vector Machine**: Margin-based, good for binary classification
6. **XGBoost**: Gradient boosting, often superior performance, handles imbalance well

### Training Strategy
- Train on preprocessed, scaled training data
- Use default hyperparameters for baseline comparison
- Prepare for hyperparameter tuning in Phase 7

In [ ]:
# Phase 5: Train all baseline models
trained_models = train_models(X_train, y_train)

print("\n" + "="*60)
print("MODELS TRAINED")
print("="*60)
for model_name in trained_models.keys():
    print(f"✓ {model_name}")

## Phase 6: Evaluate Model Performance

Evaluate all trained models on the test set using multiple metrics:

### Metrics Used
- **Accuracy**: Overall correctness (can be misleading with imbalanced data)
- **Precision**: True Positives / (True Positives + False Positives) - Avoid false alarms
- **Recall**: True Positives / (True Positives + False Negatives) - Catch all positives
- **F1 Score**: Harmonic mean of Precision and Recall - Balanced metric
- **ROC-AUC**: Area under the ROC curve - Measures discrimination ability
- **Confusion Matrix**: Shows TN, FP, FN, TP distribution

In [ ]:
# Phase 6: Evaluate all models
results_df, predictions_dict, probabilities_dict, confusion_matrices = evaluate_models(
    trained_models, X_test, y_test
)

## Phase 7: Hyperparameter Tuning

Optimize hyperparameters for the two most flexible models:

### Random Forest Tuning
- **n_estimators**: [50, 100, 200] - Number of trees in ensemble
- **max_depth**: [5, 10, 15] - Maximum depth of trees
- **min_samples_split**: [2, 5, 10] - Minimum samples to split nodes

### XGBoost Tuning
- **learning_rate**: [0.01, 0.05, 0.1] - Shrinkage (lower = slower learning)
- **max_depth**: [3, 5, 7] - Maximum tree depth
- **n_estimators**: [50, 100, 150] - Number of boosting rounds
- **subsample**: [0.7, 0.8, 1.0] - Fraction of samples for each tree

### Tuning Strategy
- Use 5-fold Cross-Validation (prevents overfitting to validation set)
- Optimize for F1 Score (balanced metric for imbalanced data)
- GridSearchCV evaluates all parameter combinations exhaustively

In [ ]:
# Phase 7: Hyperparameter tuning (this may take a few minutes)
best_rf, best_xgb, tuning_results = hyperparameter_tuning(X_train, y_train)

## Leaderboard: All Models Comparison

Create comprehensive leaderboard comparing baseline and tuned models on test set performance.

In [ ]:
# Create comprehensive leaderboard
leaderboard, champion_model = create_results_leaderboard(
    results_df, trained_models, best_rf, best_xgb, X_test, y_test
)

# Save leaderboard
leaderboard.to_csv('models/leaderboard.csv', index=False)
print("\n✓ Leaderboard saved to: models/leaderboard.csv")

## Visualization 1: Accuracy vs F1 Score Comparison

Compare key performance metrics across all models to identify the best performers.
F1 Score is more reliable than Accuracy for imbalanced datasets.

In [ ]:
# Visualization 1: Accuracy vs F1 Score
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(leaderboard))
width = 0.35

bars1 = ax.bar(x - width/2, leaderboard['Accuracy'], width, label='Accuracy', color='skyblue', edgecolor='black')
bars2 = ax.bar(x + width/2, leaderboard['F1 Score'], width, label='F1 Score', color='coral', edgecolor='black')

ax.set_xlabel('Model', fontweight='bold', fontsize=12)
ax.set_ylabel('Score', fontweight='bold', fontsize=12)
ax.set_title('Model Comparison: Accuracy vs F1 Score', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(leaderboard['Model'], rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}',
               ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('visuals/06_accuracy_vs_f1_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: visuals/06_accuracy_vs_f1_comparison.png")

## Visualization 2: Superimposed ROC Curves

ROC curves show the trade-off between True Positive Rate and False Positive Rate.
Area Under the Curve (AUC) measures model discrimination ability (higher is better).
AUC = 0.5 means random classifier, AUC = 1.0 means perfect classifier.

In [ ]:
# Visualization 2: Superimposed ROC Curves
fig, ax = plt.subplots(figsize=(10, 8))

# Define colors for different models
colors = plt.cm.tab10(np.linspace(0, 1, len(trained_models) + 2))
color_idx = 0

# Plot ROC curves for baseline models
for model_name, model in trained_models.items():
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)
    ax.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})', 
           linewidth=2.5, color=colors[color_idx])
    color_idx += 1

# Plot ROC curves for tuned models
y_pred_proba_rf = best_rf.predict_proba(X_test)[:, 1]
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)
auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
ax.plot(fpr_rf, tpr_rf, label=f'Random Forest (Tuned) (AUC={auc_rf:.3f})',
       linewidth=2.5, linestyle='--', color=colors[color_idx])
color_idx += 1

y_pred_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_pred_proba_xgb)
auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
ax.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (Tuned) (AUC={auc_xgb:.3f})',
       linewidth=2.5, linestyle='--', color=colors[color_idx])

# Plot random classifier line
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier (AUC=0.500)')

ax.set_xlabel('False Positive Rate', fontweight='bold', fontsize=12)
ax.set_ylabel('True Positive Rate', fontweight='bold', fontsize=12)
ax.set_title('ROC Curves - All Models Comparison', fontweight='bold', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('visuals/07_roc_curves_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: visuals/07_roc_curves_comparison.png")

## Visualization 3: Confusion Matrix Grid (2x3 Layout)

Confusion matrices show the distribution of predictions:
- **True Negatives (TN)**: Correctly predicted No Churn
- **False Positives (FP)**: Incorrectly predicted Churn (Type I error)
- **False Negatives (FN)**: Incorrectly predicted No Churn (Type II error)  
- **True Positives (TP)**: Correctly predicted Churn

In [ ]:
# Visualization 3: Confusion Matrix Grid (2x3 layout)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

model_names = list(trained_models.keys())
for idx, model_name in enumerate(model_names):
    cm = confusion_matrices[model_name]
    
    # Plot confusion matrix as heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
               cbar=False, square=True, annot_kws={'fontsize': 11, 'fontweight': 'bold'})
    
    axes[idx].set_title(f'{model_name}', fontweight='bold', fontsize=12)
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_xticklabels(['No Churn', 'Churn'])
    axes[idx].set_yticklabels(['No Churn', 'Churn'])

plt.suptitle('Confusion Matrices - All Baseline Models (2x3 Grid)', 
            fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('visuals/08_confusion_matrices_grid.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: visuals/08_confusion_matrices_grid.png")

## Visualization 4: Precision-Recall Curves for Top 3 Models

Precision-Recall curves are particularly useful for imbalanced datasets.
They show the trade-off between finding positive cases (recall) and precision.
Area Under the PR Curve (AUPRC) indicates overall performance (higher is better).

In [ ]:
# Visualization 4: Precision-Recall Curves for Top 3 Models
fig, ax = plt.subplots(figsize=(10, 8))

# Get top 3 models by F1 Score
top_3_models = leaderboard.head(3)['Model'].tolist()

colors = plt.cm.Set1(np.linspace(0, 1, 3))

for idx, model_name in enumerate(top_3_models):
    if model_name in trained_models:
        model = trained_models[model_name]
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    elif 'Random Forest (Tuned)' in model_name:
        y_pred_proba = best_rf.predict_proba(X_test)[:, 1]
    elif 'XGBoost (Tuned)' in model_name:
        y_pred_proba = best_xgb.predict_proba(X_test)[:, 1]
    else:
        continue
    
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    auprc = np.trapz(precision[::-1], recall[::-1])
    
    ax.plot(recall, precision, marker='o', markersize=4, linewidth=2.5,
           label=f'{model_name} (AUPRC={auprc:.3f})', color=colors[idx])

ax.set_xlabel('Recall (True Positive Rate)', fontweight='bold', fontsize=12)
ax.set_ylabel('Precision', fontweight='bold', fontsize=12)
ax.set_title('Precision-Recall Curves - Top 3 Models', fontweight='bold', fontsize=14)
ax.legend(loc='best', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('visuals/09_precision_recall_top3.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: visuals/09_precision_recall_top3.png")

## Visualization 5: Feature Importance for Champion Model

Feature importance shows which features have the most impact on the champion model's predictions.
Higher importance values indicate stronger influence on churn predictions.

In [ ]:
# Visualization 5: Feature Importance for Champion Model
# Determine champion model and get feature importances
if 'Random Forest' in champion_model:
    if 'Tuned' in champion_model:
        champion_estimator = best_rf
    else:
        champion_estimator = trained_models['Random Forest']
    feature_importance = champion_estimator.feature_importances_
elif 'XGBoost' in champion_model:
    if 'Tuned' in champion_model:
        champion_estimator = best_xgb
    else:
        champion_estimator = trained_models['XGBoost']
    feature_importance = champion_estimator.feature_importances_
else:
    # For other models, use permutation importance
    from sklearn.inspection import permutation_importance
    perm_importance = permutation_importance(
        trained_models.get(champion_model, trained_models['Random Forest']),
        X_test, y_test, n_repeats=10, random_state=42
    )
    feature_importance = perm_importance.importances_mean

# Create feature importance dataframe
feature_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False).head(15)

# Plot feature importance
fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(feature_importance_df)), feature_importance_df['Importance'], 
               color='steelblue', edgecolor='black', linewidth=1.2)

ax.set_yticks(range(len(feature_importance_df)))
ax.set_yticklabels(feature_importance_df['Feature'])
ax.set_xlabel('Importance Score', fontweight='bold', fontsize=12)
ax.set_title(f'Top 15 Features - {champion_model} (Champion Model)', 
            fontweight='bold', fontsize=14)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2.,
           f'{width:.4f}',
           ha='left', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('visuals/10_feature_importance_champion.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: visuals/10_feature_importance_champion.png")

## Save All Models

Serialize all trained models (both baseline and tuned) for future use and deployment.

In [ ]:
# Save all models
save_models(trained_models, best_rf, best_xgb)

print("\n" + "="*60)
print("MODELS SAVED SUCCESSFULLY")
print("="*60)
print("\nModel Files Created:")
print("  Baseline Models:")
for model_name in trained_models.keys():
    filename = f"models/{model_name.lower().replace(' ', '_')}_baseline.pkl"
    print(f"    ✓ {filename}")
print("  Tuned Models:")
print("    ✓ models/random_forest_tuned.pkl")
print("    ✓ models/xgboost_tuned.pkl")

## Summary: Model Training & Evaluation Complete! 🎉

### ✓ Completed Tasks

1. **Phase 5 - Model Training**
   - Trained 6 classification models with default hyperparameters
   - Logistic Regression, Decision Tree, Random Forest, KNN, SVM, XGBoost
   
2. **Phase 6 - Model Evaluation**
   - Evaluated all models on test set (1,409 samples)
   - Calculated 5 key metrics: Accuracy, Precision, Recall, F1 Score, ROC-AUC
   - Generated confusion matrices for error analysis
   
3. **Phase 7 - Hyperparameter Tuning**
   - Optimized Random Forest with 3×3×3 = 27 combinations
   - Optimized XGBoost with 3×3×3×3 = 81 combinations
   - Used 5-fold Cross-Validation with F1 Score optimization

### 📊 Key Results

- **Champion Model**: {{champion_model}}
- **Champion F1 Score**: {{leaderboard.iloc[0]['F1 Score']:.4f}}
- **Champion ROC-AUC**: {{leaderboard.iloc[0]['ROC-AUC']:.4f}}

### 📁 Generated Files

**Visualizations** (saved to `visuals/`):
- 06_accuracy_vs_f1_comparison.png
- 07_roc_curves_comparison.png
- 08_confusion_matrices_grid.png
- 09_precision_recall_top3.png
- 10_feature_importance_champion.png

**Models** (saved to `models/`):
- Baseline models: 6 pkl files
- Tuned models: 2 pkl files
- leaderboard.csv: Complete results comparison

### 🚀 Next Steps

1. **Deploy Champion Model**: Use {{champion_model}} for production predictions
2. **Monitor Performance**: Track model performance over time
3. **Handle Class Imbalance**: Consider SMOTE or class weights if needed
4. **Ensemble Methods**: Combine top 3 models for even better performance
5. **A/B Testing**: Test champion model against baseline in production

### 📈 Model Performance Insights

- **Best for Recall** (catch all churners): Model with highest Recall
- **Best for Precision** (avoid false alarms): Model with highest Precision
- **Best Balance**: Champion Model (highest F1 Score)
- **Best Discrimination**: Model with highest ROC-AUC

### 💡 Recommendations

1. Deploy {{champion_model}} to production
2. Retrain quarterly on new data
3. Monitor for concept drift
4. Consider ensemble prediction combining top 3 models
5. Implement feature monitoring to detect data drift